In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *
from view import *
from reprojection import *


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])


#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,30)], axis=0))[0] # CR2314

t = np.where(np.all([dates > datetime(2026,8,30),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260830T225003_V202610191352C_0648300501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260928T093003_V202610191523C_0649280504.fits'),
 85)

In [3]:
data, view = make_map(files[:],
                      correct_dr=True,
                      kind='synoptic',
                      correct_bias=False,
                      )

/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:98: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))
/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:135: RuntimeWarning: invalid value encountered in divide
  S = (FF - GF[:, :, -1:, :] ** 2 - Q ** 2 * (LL - GL[:, :, -1:, :] ** 2)) / LL / np.expand_dims(N - 1, (-1, -2))


In [4]:
show_data(data, view, vmin=-500, vmax=500)

/home/ulyanov/PycharmProjects/reprojection/src/show.py:125: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


(<Figure size 1500x600 with 2 Axes>,
 <Axes: xlabel='Longitude, deg', ylabel='Latitude, deg'>)

In [5]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2315.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

#data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.09.12_09:28:02_TAI'

In [6]:
show_data(data_, view, vmin=-500, vmax=500)

(<Figure size 1500x600 with 2 Axes>,
 <Axes: xlabel='Longitude, deg', ylabel='Latitude, deg'>)

In [7]:
s = np.load('/home/ulyanov/data/sdo/hmi/sft1d.npz')
flux_sft = s['fluxes'][-1]
lat_sft = s['lat']

In [10]:
nx = len(data)
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
flux = np.nanmean(data, axis=1)

nx = len(data_)
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
flux_hmi = np.nanmean(data_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat_sft, flux_sft, label='SFT')
plt.plot(lat, flux, label='FDT')


plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_162273/437853492.py:3: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(data, axis=1)
/tmp/ipykernel_162273/437853492.py:7: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)


In [9]:
np.mean(flux) / np.mean(np.abs(flux)), np.mean(data) / np.mean(np.abs(data))

(np.float64(nan), np.float64(nan))

In [3]:
np.mean(flux_sft) / np.mean(np.abs(flux_sft))

np.float64(-0.1177986881831943)

In [ ]:
np.savez('map.npz', data=data)